
# PLAYER MONITORING




# Objective

To develop a sports video analytics system that detects and tracks players and the ball using YOLOv8, and generates movement trajectories, distance covered, and a heatmap for performance analysis.



In [39]:
# =====================================
# Install libraries
# =====================================
!pip -q install ultralytics opencv-python-headless

# =====================================
# Imports
# =====================================
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from google.colab import files


In [40]:
video_path = "/content/Players_Juggling.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(3))
height = int(cap.get(4))
fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))

print("Width:", width)
print("Height:", height)
print("FPS:", fps)


Width: 1920
Height: 1080
FPS: 60


In [41]:
model = YOLO("yolov8s.pt")


In [42]:
out = cv2.VideoWriter(
    "analytics_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)


In [43]:
heatmap = np.zeros((height, width), dtype=np.float32)

tracks = {}
next_id = 0

# 🔥 ADD THESE
trajectory = {}
distance_covered = {}


In [44]:
def centroid(box):
    x1,y1,x2,y2 = box
    return int((x1+x2)/2), int((y1+y2)/2)


In [45]:
def match_object(c):
    global next_id
    for obj_id, prev in tracks.items():
        if np.linalg.norm(np.array(c)-np.array(prev)) < 50:
            return obj_id
    tracks[next_id] = c
    next_id += 1
    return next_id-1


In [46]:
frame_count = 0

while cap.isOpened():

    ret, frame = cap.read()
    if not ret or frame is None:
        print("End of video")
        break

    people_count = 0

    # Fade heatmap for realistic effect
    heatmap *= 0.95

    results = model(frame, verbose=False)

    for r in results:
        for box in r.boxes.data:

            x1, y1, x2, y2, conf, cls = box.tolist()
            cls = int(cls)

            # ===============================
            # PLAYER TRACKING (class 0)
            # ===============================
            if cls == 0:

                people_count += 1

                c = centroid((x1, y1, x2, y2))
                obj_id = match_object(c)

                # Initialize structures
                if obj_id not in trajectory:
                    trajectory[obj_id] = []
                    distance_covered[obj_id] = 0

                # Compute distance
                if len(trajectory[obj_id]) > 0:
                    prev = trajectory[obj_id][-1]
                    dist = np.linalg.norm(np.array(c) - np.array(prev))
                    distance_covered[obj_id] += dist

                trajectory[obj_id].append(c)
                tracks[obj_id] = c

                # Update heatmap
                cv2.circle(heatmap, c, 20, 1, -1)

                # Draw bounding box
                cv2.rectangle(frame,
                              (int(x1), int(y1)),
                              (int(x2), int(y2)),
                              (0, 255, 0), 2)

                # Draw ID + distance
                cv2.putText(frame,
                            f"ID {obj_id} | Dist {int(distance_covered[obj_id])}",
                            (int(x1), int(y1)-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5,
                            (0, 0, 255),
                            2)

                # Draw trajectory lines
                for i in range(1, len(trajectory[obj_id])):
                    cv2.line(frame,
                             trajectory[obj_id][i-1],
                             trajectory[obj_id][i],
                             (255, 0, 0), 2)

            # ===============================
            # BALL TRACKING (class 32)
            # ===============================
            if cls == 32:

                c_ball = centroid((x1, y1, x2, y2))

                # Draw ball differently
                cv2.rectangle(frame,
                              (int(x1), int(y1)),
                              (int(x2), int(y2)),
                              (0, 165, 255), 2)

                cv2.putText(frame,
                            "BALL",
                            c_ball,
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,
                            (0, 165, 255),
                            2)

                # Draw ball center
                cv2.circle(frame, c_ball, 5, (0, 165, 255), -1)


    # ===============================
    # Heatmap Overlay
    # ===============================
    hm = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX)
    hm = hm.astype(np.uint8)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)

    if hm_color.shape[:2] != frame.shape[:2]:
        hm_color = cv2.resize(hm_color,
                              (frame.shape[1], frame.shape[0]))

    overlay = cv2.addWeighted(frame, 0.7,
                              hm_color, 0.3,
                              0)

    # Add player count
    cv2.putText(overlay,
                f"Players: {people_count}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2)

    out.write(overlay)

    frame_count += 1

    if frame_count % 100 == 0:
        print("Processed:", frame_count)


cap.release()
out.release()

print("✅ Advanced Sports Analytics Complete!")


Processed: 100
Processed: 200
Processed: 300
Processed: 400
Processed: 500
Processed: 600
Processed: 700
Processed: 800
End of video
✅ Advanced Sports Analytics Complete!


In [47]:
from google.colab import files
files.download("analytics_output.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



# Observation

During the execution of the practical, the YOLOv8 model successfully detected players in most frames with good accuracy. Each detected player was assigned a unique ID using centroid-based tracking. The tracking system maintained consistent IDs when player motion was smooth and there was minimal occlusion.

Player trajectories were visualized using line paths drawn across consecutive frames, clearly showing movement patterns and direction of play. The total distance covered by each player was computed using Euclidean distance between centroids in successive frames (measured in pixel units).

The sports ball was detected when clearly visible; however, detection accuracy decreased when the ball appeared small or partially occluded. The generated heatmap effectively highlighted high-activity areas on the field, and the fading mechanism produced a dynamic representation of movement intensity over time.

---

# Conclusion

The practical successfully implemented a complete sports analytics pipeline by combining deep learning-based object detection with classical tracking techniques. The system effectively analyzed player movement, visualized activity patterns, and demonstrated the practical application of computer vision in sports performance analysis.
